# Premier League ML Predictor — Data Pipeline

## 1. Imports & Setup

In [1]:
import requests
import pandas as pd
from io import StringIO


#Makes sure can see all columns and 50 rows before truncating
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)

## 2. Data Collection

Source: [football-data.co.uk](https://www.football-data.co.uk/englandm.php)  
URL pattern: `https://www.football-data.co.uk/mmz4281/{YYYY}/E0.csv`  
Columns we keep: `Date, HomeTeam, AwayTeam, FTHG, FTAG, FTR, HS, AS, HST, AST, HC, AC`

FTHG — Full Time Home Goals
FTAG — Full Time Away Goals
FTR — Full Time Result (H/D/A)
HS — Home Shots
AS — Away Shots
HST — Home Shots on Target
AST — Away Shots on Target
HC — Home Corners
AC — Away Corners

In [53]:
# Seasons to collect: (display_name, url_code)
SEASONS = [
    ("2014-15", "1415"),
    ("2015-16", "1516"),
    ("2016-17", "1617"),
    ("2017-18", "1718"),
    ("2018-19", "1819"),
    ("2019-20", "1920"),
    ("2020-21", "2021"),
    ("2021-22", "2122"),
    ("2022-23", "2223"),
    ("2023-24", "2324"),
    ("2024-25", "2425"),
]

COLS = ["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR", "HS", "AS", "HST", "AST", "HC", "AC"]

all_seasons = []

for season_name, code in SEASONS:
    url = f"https://www.football-data.co.uk/mmz4281/{code}/E0.csv"
    print(f"Downloading {season_name}... ", end="")
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        df = pd.read_csv(StringIO(response.text), usecols=COLS)
        df["season"] = season_name
        all_seasons.append(df)
        print(f"{len(df)} rows")
    except Exception as e:
        print(f"FAILED: {e}")

raw = pd.concat(all_seasons, ignore_index=True)
print(f"\nTotal rows: {len(raw)}")


Total rows: 4181


In [54]:
raw.head(10)

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HS,AS,HST,AST,HC,AC,season
0,16/08/14,Arsenal,Crystal Palace,2.0,1.0,H,14.0,4.0,6.0,2.0,9.0,3.0,2014-15
1,16/08/14,Leicester,Everton,2.0,2.0,D,11.0,13.0,3.0,3.0,3.0,6.0,2014-15
2,16/08/14,Man United,Swansea,1.0,2.0,A,14.0,5.0,5.0,4.0,4.0,0.0,2014-15
3,16/08/14,QPR,Hull,0.0,1.0,A,19.0,11.0,6.0,4.0,8.0,9.0,2014-15
4,16/08/14,Stoke,Aston Villa,0.0,1.0,A,12.0,7.0,2.0,2.0,2.0,8.0,2014-15
5,16/08/14,West Brom,Sunderland,2.0,2.0,D,10.0,7.0,5.0,2.0,6.0,3.0,2014-15
6,16/08/14,West Ham,Tottenham,0.0,1.0,A,18.0,10.0,4.0,4.0,8.0,5.0,2014-15
7,17/08/14,Liverpool,Southampton,2.0,1.0,H,12.0,12.0,5.0,6.0,2.0,6.0,2014-15
8,17/08/14,Newcastle,Man City,0.0,2.0,A,12.0,13.0,0.0,5.0,3.0,3.0,2014-15
9,18/08/14,Burnley,Chelsea,1.0,3.0,A,9.0,11.0,2.0,3.0,4.0,3.0,2014-15


In [55]:
raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4181 entries, 0 to 4180
Data columns (total 13 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Date      4180 non-null   object 
 1   HomeTeam  4180 non-null   object 
 2   AwayTeam  4180 non-null   object 
 3   FTHG      4180 non-null   float64
 4   FTAG      4180 non-null   float64
 5   FTR       4180 non-null   object 
 6   HS        4180 non-null   float64
 7   AS        4180 non-null   float64
 8   HST       4180 non-null   float64
 9   AST       4180 non-null   float64
 10  HC        4180 non-null   float64
 11  AC        4180 non-null   float64
 12  season    4181 non-null   object 
dtypes: float64(8), object(5)
memory usage: 424.8+ KB


In [56]:
# check for missing values per column
raw.isnull().sum()

Date        1
HomeTeam    1
AwayTeam    1
FTHG        1
FTAG        1
FTR         1
HS          1
AS          1
HST         1
AST         1
HC          1
AC          1
season      0
dtype: int64

In [57]:
# All unique team names — check for inconsistencies
all_teams = pd.Series(
    pd.concat([raw["HomeTeam"], raw["AwayTeam"]]).unique()
).sort_values().reset_index(drop=True)
print(all_teams.tolist())

['Arsenal', 'Aston Villa', 'Bournemouth', 'Brentford', 'Brighton', 'Burnley', 'Cardiff', 'Chelsea', 'Crystal Palace', 'Everton', 'Fulham', 'Huddersfield', 'Hull', 'Ipswich', 'Leeds', 'Leicester', 'Liverpool', 'Luton', 'Man City', 'Man United', 'Middlesbrough', 'Newcastle', 'Norwich', "Nott'm Forest", 'QPR', 'Sheffield United', 'Southampton', 'Stoke', 'Sunderland', 'Swansea', 'Tottenham', 'Watford', 'West Brom', 'West Ham', 'Wolves', nan]


In [58]:
raw[raw.isnull().any(axis=1)]
#the 2014-15 season has an extra row at the end with just nulls we can drop it. 

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HS,AS,HST,AST,HC,AC,season
380,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2014-15


In [59]:
df = raw.dropna(subset=["HomeTeam"]).copy()
print(len(df))  # should be 4180 total rows after dropping one


4180


In [60]:
df.head(383)

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HS,AS,HST,AST,HC,AC,season
0,16/08/14,Arsenal,Crystal Palace,2.0,1.0,H,14.0,4.0,6.0,2.0,9.0,3.0,2014-15
1,16/08/14,Leicester,Everton,2.0,2.0,D,11.0,13.0,3.0,3.0,3.0,6.0,2014-15
2,16/08/14,Man United,Swansea,1.0,2.0,A,14.0,5.0,5.0,4.0,4.0,0.0,2014-15
3,16/08/14,QPR,Hull,0.0,1.0,A,19.0,11.0,6.0,4.0,8.0,9.0,2014-15
4,16/08/14,Stoke,Aston Villa,0.0,1.0,A,12.0,7.0,2.0,2.0,2.0,8.0,2014-15
...,...,...,...,...,...,...,...,...,...,...,...,...,...
378,24/05/15,Newcastle,West Ham,2.0,0.0,H,17.0,4.0,4.0,1.0,2.0,3.0,2014-15
379,24/05/15,Stoke,Liverpool,6.0,1.0,H,15.0,13.0,9.0,4.0,3.0,9.0,2014-15
381,08/08/2015,Bournemouth,Aston Villa,0.0,1.0,A,11.0,7.0,2.0,3.0,6.0,3.0,2015-16
382,08/08/2015,Chelsea,Swansea,2.0,2.0,D,11.0,18.0,3.0,10.0,4.0,8.0,2015-16


Data Cleaning

In [136]:
# Parse date, fix dtypes, sort chronologically
df = raw.dropna(subset=["HomeTeam"]).copy()
df["Date"] = pd.to_datetime(df["Date"], format='mixed', dayfirst=True)
df["FTHG"] = df["FTHG"].astype(int)
df["FTAG"] = df["FTAG"].astype(int)
df = df.sort_values("Date").reset_index(drop=True)

df.head(10)


,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HS,AS,HST,AST,HC,AC,season
0,2014-08-16,Arsenal,Crystal Palace,2,1,H,14.0,4.0,6.0,2.0,9.0,3.0,2014-15
1,2014-08-16,Leicester,Everton,2,2,D,11.0,13.0,3.0,3.0,3.0,6.0,2014-15
2,2014-08-16,Man United,Swansea,1,2,A,14.0,5.0,5.0,4.0,4.0,0.0,2014-15
3,2014-08-16,QPR,Hull,0,1,A,19.0,11.0,6.0,4.0,8.0,9.0,2014-15
4,2014-08-16,Stoke,Aston Villa,0,1,A,12.0,7.0,2.0,2.0,2.0,8.0,2014-15
5,2014-08-16,West Brom,Sunderland,2,2,D,10.0,7.0,5.0,2.0,6.0,3.0,2014-15
6,2014-08-16,West Ham,Tottenham,0,1,A,18.0,10.0,4.0,4.0,8.0,5.0,2014-15
7,2014-08-17,Newcastle,Man City,0,2,A,12.0,13.0,0.0,5.0,3.0,3.0,2014-15
8,2014-08-17,Liverpool,Southampton,2,1,H,12.0,12.0,5.0,6.0,2.0,6.0,2014-15
9,2014-08-18,Burnley,Chelsea,1,3,A,9.0,11.0,2.0,3.0,4.0,3.0,2014-15


## Feature Engineering

Rolling averages: home/away split (last 5, min_periods=3) 

In [137]:
# Home team stats from their previous HOME matches
home_rolling = (
    df[["Date", "HomeTeam", "FTHG", "FTAG", "HS", "HST", "HC"]]
    .copy()
    .sort_values("Date")
    .groupby("HomeTeam")[["FTHG", "FTAG", "HS", "HST", "HC"]]
    .apply(lambda x: x.shift(1).rolling(5, min_periods=3).mean())
    .reset_index(level=0, drop=True)
    .rename(columns={
        "FTHG": "home_rolling5_goals_scored",
        "FTAG": "home_rolling5_goals_conceded",
        "HS":   "home_rolling5_shots",
        "HST":  "home_rolling5_shots_on_target",
        "HC":   "home_rolling5_corners",
    })
)

# Away team stats from their previous AWAY matches
away_rolling = (
    df[["Date", "AwayTeam", "FTAG", "FTHG", "AS", "AST", "AC"]]
    .copy()
    .sort_values("Date")
    .groupby("AwayTeam")[["FTAG", "FTHG", "AS", "AST", "AC"]]
    .apply(lambda x: x.shift(1).rolling(5, min_periods=3).mean())
    .reset_index(level=0, drop=True)
    .rename(columns={
        "FTAG": "away_rolling5_goals_scored",
        "FTHG": "away_rolling5_goals_conceded",
        "AS":   "away_rolling5_shots",
        "AST":  "away_rolling5_shots_on_target",
        "AC":   "away_rolling5_corners",
    })
)

df = df.join(home_rolling).join(away_rolling)
df[["HomeTeam", "Date", "home_rolling5_goals_scored", "away_rolling5_goals_scored"]].head(100)


,HomeTeam,Date,home_rolling5_goals_scored,away_rolling5_goals_scored
0,Arsenal,2014-08-16,NaN,NaN
1,Leicester,2014-08-16,NaN,NaN
2,Man United,2014-08-16,NaN,NaN
3,QPR,2014-08-16,NaN,NaN
4,Stoke,2014-08-16,NaN,NaN
...,...,...,...,...
95,Newcastle,2014-11-01,1.50,2.00
96,Stoke,2014-11-01,0.75,2.25
97,Aston Villa,2014-11-02,0.50,1.25
98,Man City,2014-11-02,2.00,1.50


Overall form given last 5 matches regardless of whether the game is home or away. 
The df has one row per match, but we need one row per team per match.
So we create two views, one from the home team's perspective and one from away team. 


In [138]:
# df copy of the home teams perspective, they win if the FTR is H
home_view = df[["Date", "HomeTeam", "FTHG", "FTAG", "FTR"]].copy()
home_view["team"] = home_view["HomeTeam"]
home_view["scored"] = home_view["FTHG"]
#map the points a team gets 3 for a win 1 for a draw and 0 for a loss
home_view["points"] = home_view["FTR"].map({"H": 3, "D": 1, "A": 0})

# df copy of the aways teams perspective, they win if FTR is A
away_view = df[["Date", "AwayTeam", "FTAG", "FTHG", "FTR"]].copy()
away_view["team"] = away_view["AwayTeam"]
away_view["scored"] = away_view["FTAG"]
away_view["points"] = away_view["FTR"].map({"A": 3, "D": 1, "H": 0})

#stack them now each match appears twice once for the home team and one for the away team
team_matches = pd.concat([
    home_view[["Date", "team", "scored", "points"]],
    away_view[["Date", "team", "scored", "points"]]
]).sort_values("Date").reset_index(drop=True)

# For each team, compute rolling avg of last 5 matches
team_matches["form_points"] = team_matches.groupby("team")["points"].transform(
    #shift(1) excludes the current match, min_periods=3 means we need at least 3 past matches to get a value, otherwise it will be NaN
    lambda x: x.shift(1).rolling(5, min_periods=3).mean()
)
team_matches["form_goals_scored"] = team_matches.groupby("team")["scored"].transform(
    lambda x: x.shift(1).rolling(5, min_periods=3).mean()
)

# Merge back for home and away
df = df.merge(
    team_matches[["Date", "team", "form_points", "form_goals_scored"]].rename(columns={
        "team": "HomeTeam", "form_points": "home_form_points", "form_goals_scored": "home_form_goals_scored"
    }),
    on=["Date", "HomeTeam"], how="left"
)
df = df.merge(
    team_matches[["Date", "team", "form_points", "form_goals_scored"]].rename(columns={
        "team": "AwayTeam", "form_points": "away_form_points", "form_goals_scored": "away_form_goals_scored"
    }),
    on=["Date", "AwayTeam"], how="left"
)

df[["HomeTeam", "AwayTeam", "Date", "home_form_points", "away_form_points"]].head(50)


,HomeTeam,AwayTeam,Date,home_form_points,away_form_points
0,Arsenal,Crystal Palace,2014-08-16,NaN,NaN
1,Leicester,Everton,2014-08-16,NaN,NaN
2,Man United,Swansea,2014-08-16,NaN,NaN
3,QPR,Hull,2014-08-16,NaN,NaN
4,Stoke,Aston Villa,2014-08-16,NaN,NaN
5,West Brom,Sunderland,2014-08-16,NaN,NaN
6,West Ham,Tottenham,2014-08-16,NaN,NaN
7,Newcastle,Man City,2014-08-17,NaN,NaN
8,Liverpool,Southampton,2014-08-17,NaN,NaN
9,Burnley,Chelsea,2014-08-18,NaN,NaN


Season cumulative averages (season_avg_scored is the average goals scored by a team in all their matches so far that season, not including the current match)
Groups by season and team so the average resets at the start of each new season

In [139]:
# Home team's perspective
home_season = df[["Date", "season", "HomeTeam", "FTHG", "FTAG"]].copy()
home_season.columns = ["Date", "season", "team", "scored", "conceded"]

# away team's perspective
away_season = df[["Date", "season", "AwayTeam", "FTAG", "FTHG"]].copy()
away_season.columns = ["Date", "season", "team", "scored", "conceded"]

#sorted chronologically and stacked so each row is a team in a season with their scored and conceded goals for that match
team_season = pd.concat([home_season, away_season]).sort_values(["team", "season", "Date"]).reset_index(drop=True)

# Expanding mean per team per season, shift(1) to exclude current match
team_season["season_avg_scored"] = team_season.groupby(["team", "season"])["scored"].transform(
    lambda x: x.shift(1).expanding().mean()
)
team_season["season_avg_conceded"] = team_season.groupby(["team", "season"])["conceded"].transform(
    lambda x: x.shift(1).expanding().mean()
)

# Merge back for home and away
# Match each row in df to the correct home team's season stats using date + season + team name
df = df.merge(
    team_season[["Date", "season", "team", "season_avg_scored", "season_avg_conceded"]].rename(columns={
        "team": "HomeTeam", "season_avg_scored": "home_season_avg_scored", "season_avg_conceded": "home_season_avg_conceded"
    }),
    # merge on date season and team name to get the correct season averages for that match, left join to keep all matches even if some teams don't have season averages yet
    on=["Date", "season", "HomeTeam"], how="left"
)
df = df.merge(
    team_season[["Date", "season", "team", "season_avg_scored", "season_avg_conceded"]].rename(columns={
        "team": "AwayTeam", "season_avg_scored": "away_season_avg_scored", "season_avg_conceded": "away_season_avg_conceded"
    }),
    on=["Date", "season", "AwayTeam"], how="left"
)

df[["HomeTeam", "AwayTeam", "season", "Date", "home_season_avg_scored", "away_season_avg_scored"]].head(60)


,HomeTeam,AwayTeam,season,Date,home_season_avg_scored,away_season_avg_scored
0,Arsenal,Crystal Palace,2014-15,2014-08-16,NaN,NaN
1,Leicester,Everton,2014-15,2014-08-16,NaN,NaN
2,Man United,Swansea,2014-15,2014-08-16,NaN,NaN
3,QPR,Hull,2014-15,2014-08-16,NaN,NaN
4,Stoke,Aston Villa,2014-15,2014-08-16,NaN,NaN
...,...,...,...,...,...,...
55,Crystal Palace,Leicester,2014-15,2014-09-27,1.6,1.8
56,Arsenal,Tottenham,2014-15,2014-09-27,2.0,1.4
57,Hull,Man City,2014-15,2014-09-27,1.4,1.6
58,West Brom,Burnley,2014-15,2014-09-28,0.6,0.2


League standings (points and position before each match)

In [140]:
# --- League standings (position and points before each gameweek) ---

# Assign gameweek: every 10 matches per season = 1 gameweek
# Note: postponed matches may slightly shift groupings, but effect is negligible
df["gameweek"] = df.groupby("season").cumcount() // 10 + 1

# Build one row per team per match with points earned
home_pts = df[["Date", "season", "gameweek", "HomeTeam", "FTR"]].copy()
home_pts.columns = ["Date", "season", "gameweek", "team", "ftr"]
home_pts["pts"] = home_pts["ftr"].map({"H": 3, "D": 1, "A": 0})

away_pts = df[["Date", "season", "gameweek", "AwayTeam", "FTR"]].copy()
away_pts.columns = ["Date", "season", "gameweek", "team", "ftr"]
away_pts["pts"] = away_pts["ftr"].map({"A": 3, "D": 1, "H": 0})

all_pts = pd.concat([home_pts, away_pts]).sort_values(["season", "gameweek"]).reset_index(drop=True)

# Cumulative points per team per season
all_pts["cumulative_points"] = all_pts.groupby(["team", "season"])["pts"].cumsum()

# Get each team's total points at the end of each gameweek
gw_standings = all_pts.groupby(["season", "gameweek", "team"])["cumulative_points"].last().reset_index()

# Rank all 20 teams within each season + gameweek
gw_standings["league_position"] = gw_standings.groupby(["season", "gameweek"])["cumulative_points"].rank(
    ascending=False, method="min"
).astype(int)

# Shift by 1 gameweek so we get standings BEFORE the current match
gw_standings["gameweek"] = gw_standings["gameweek"] + 1

# Merge back for home team
df = df.merge(
    gw_standings.rename(columns={
        "team": "HomeTeam", "cumulative_points": "home_points", "league_position": "home_league_position"
    }),
    on=["season", "gameweek", "HomeTeam"], how="left"
)
# Merge back for away team
df = df.merge(
    gw_standings.rename(columns={
        "team": "AwayTeam", "cumulative_points": "away_points", "league_position": "away_league_position"
    }),
    on=["season", "gameweek", "AwayTeam"], how="left"
)

#convert NAN's to 0 and make point and league position as an int instead of a float. 
df["home_points"] = df["home_points"].fillna(0).astype(int)
df["away_points"] = df["away_points"].fillna(0).astype(int)
df["home_league_position"] = df["home_league_position"].fillna(10).astype(int)
df["away_league_position"] = df["away_league_position"].fillna(10).astype(int)


df[["HomeTeam", "AwayTeam", "Date", "gameweek", "home_points", "away_points", "home_league_position", "away_league_position"]].head(30)

,HomeTeam,AwayTeam,Date,gameweek,home_points,away_points,home_league_position,away_league_position
0,Arsenal,Crystal Palace,2014-08-16,1,0,0,10,10
1,Leicester,Everton,2014-08-16,1,0,0,10,10
2,Man United,Swansea,2014-08-16,1,0,0,10,10
3,QPR,Hull,2014-08-16,1,0,0,10,10
4,Stoke,Aston Villa,2014-08-16,1,0,0,10,10
5,West Brom,Sunderland,2014-08-16,1,0,0,10,10
6,West Ham,Tottenham,2014-08-16,1,0,0,10,10
7,Newcastle,Man City,2014-08-17,1,0,0,10,10
8,Liverpool,Southampton,2014-08-17,1,0,0,10,10
9,Burnley,Chelsea,2014-08-18,1,0,0,10,10


Head to head results. For each match look at all the previous meetings get the results and the average win rate. 

In [141]:
h2h_win_rate = []
h2h_avg_total_goals = []

for i, row in df.iterrows():
    # only look at the matches before this one to avoid data leakage
    prev = df.loc[:i-1]
    #find all the prvious meetings between these two teams regardless of home/away
    meetings = prev[
        ((prev["HomeTeam"] == row["HomeTeam"]) & (prev["AwayTeam"] == row["AwayTeam"])) |
        ((prev["HomeTeam"] == row["AwayTeam"]) & (prev["AwayTeam"] == row["HomeTeam"]))
    ]
    
    if len(meetings) == 0:
        h2h_win_rate.append(None)
        h2h_avg_total_goals.append(None)
    else:
        #how often the home team won in any previous meeting
        wins_as_home = (meetings[meetings["HomeTeam"] == row["HomeTeam"]]["FTR"] == "H").sum()
        wins_as_away = (meetings[meetings["AwayTeam"] == row["HomeTeam"]]["FTR"] == "A").sum()
        total_wins = wins_as_home + wins_as_away

        #calculate the win rate as a proportion
        h2h_win_rate.append(total_wins / len(meetings))
        #calculate the average total goals scored in their meetings which helps predict high or low scoring game.
        h2h_avg_total_goals.append((meetings["FTHG"] + meetings["FTAG"]).mean())

df["h2h_win_rate"] = h2h_win_rate
df["h2h_avg_total_goals"] = h2h_avg_total_goals

df[["HomeTeam", "AwayTeam", "Date", "h2h_win_rate", "h2h_avg_total_goals"]].dropna().head(50)


,HomeTeam,AwayTeam,Date,h2h_win_rate,h2h_avg_total_goals
190,Tottenham,Chelsea,2015-01-01,0.0,3.0
191,Stoke,Man United,2015-01-01,0.0,3.0
192,West Ham,West Brom,2015-01-01,1.0,3.0
193,QPR,Swansea,2015-01-01,0.0,2.0
194,Southampton,Arsenal,2015-01-01,0.0,1.0
195,Man City,Sunderland,2015-01-01,1.0,5.0
196,Newcastle,Burnley,2015-01-01,0.0,2.0
197,Liverpool,Leicester,2015-01-01,1.0,4.0
198,Hull,Everton,2015-01-01,0.0,2.0
199,Aston Villa,Crystal Palace,2015-01-01,1.0,1.0


Derived Features.

Shot accuracy: what proportion of shots are on target (rolling avg from earlier features)

home_shot_accuracy — rolling HST/HS ratio
away_shot_accuracy — rolling AST/AS ratio
home_away_strength_diff — home_points minus away_points (Calculates the quality gaps between the teams)


In [142]:
# replace 0 shots with NaN to avoid division by zero when calculating shot accuracy
df["home_shot_accuracy"] = df["home_rolling5_shots_on_target"] / df["home_rolling5_shots"].replace(0, None)
df["away_shot_accuracy"] = df["away_rolling5_shots_on_target"] / df["away_rolling5_shots"].replace(0, None)
df["home_away_strength_diff"] = df["home_points"] - df["away_points"]

df[["HomeTeam", "AwayTeam", "Date", "home_shot_accuracy", "away_shot_accuracy", "home_away_strength_diff"]].iloc[60:80]


,HomeTeam,AwayTeam,Date,home_shot_accuracy,away_shot_accuracy,home_away_strength_diff
60,Aston Villa,Man City,2014-10-04,0.230769,0.395833,-1
61,Hull,Crystal Palace,2014-10-04,0.433333,0.461538,-2
62,Leicester,Burnley,2014-10-04,0.314286,0.181818,5
63,Liverpool,West Brom,2014-10-04,0.259259,0.235294,-1
64,Sunderland,Stoke,2014-10-04,0.218750,0.264706,-3
65,Swansea,Newcastle,2014-10-04,0.500000,0.250000,7
66,West Ham,QPR,2014-10-05,0.371429,0.178571,3
67,Tottenham,Southampton,2014-10-05,0.218750,0.439024,-5
68,Chelsea,Arsenal,2014-10-05,0.369863,0.260870,6
69,Man United,Everton,2014-10-05,0.414634,0.342857,2


In [143]:
# Rename targets 
df = df.rename(columns={"FTHG": "home_goals", "FTAG": "away_goals", "FTR": "result"})
# split the test data
df["is_test"] = df["season"] == "2024-25"

# drop the columns we used to calculate features but don't want in the final dataset
df = df.drop(columns=["HS", "AS", "HST", "AST", "HC", "AC"])

# Save to CSV
df.to_csv("pl_matches_final.csv", index=False)
print(f"Saved {len(df)} rows to pl_matches_final.csv")


Saved 4180 rows to pl_matches_final.csv
